In [0]:
CREATE OR REPLACE TABLE data_governance.gold_access.dim_users USING DELTA AS

SELECT DISTINCT
    COALESCE(a.run_by, u.initiated_by) AS user_id,
    a.account_id,
    a.workspace_id,
    CURRENT_TIMESTAMP() AS load_timestamp

FROM data_governance.silver_access_management.audit_logs a
FULL OUTER JOIN data_governance.silver_access_management.assistant_usage u
    ON a.workspace_id = u.workspace_id

In [0]:
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_access.fact_user_activity
),

new_audit AS (
    SELECT *
    FROM data_governance.silver_access_management.audit_logs
    WHERE load_timestamp > (SELECT last_ts FROM last_run)
),

new_assistant AS (
    SELECT *
    FROM data_governance.silver_access_management.assistant_usage
    WHERE load_timestamp > (SELECT last_ts FROM last_run)
),

combined AS (
    SELECT
        COALESCE(a.account_id, u.account_id) AS account_id,
        COALESCE(a.workspace_id, u.workspace_id) AS workspace_id,

        COALESCE(a.run_by, u.initiated_by) AS user_id,

        COALESCE(a.event_id, u.event_id) AS event_id,
        COALESCE(a.event_time, u.event_time) AS event_time,
        COALESCE(a.event_date, u.event_date) AS event_date,

        a.service_name,
        a.action_name,

        u.user_agent,
        u.browser,
        u.os,
        u.is_bot,

        a.session_id,
        a.source_ip_address,

        CURRENT_TIMESTAMP() AS load_timestamp

    FROM new_audit a
    FULL OUTER JOIN new_assistant u
        ON a.workspace_id = u.workspace_id
        AND a.event_date = u.event_date
)

INSERT INTO data_governance.gold_access.fact_user_activity
SELECT * FROM combined;

In [0]:
CREATE OR REPLACE TABLE data_governance.gold_access.fact_table_access
USING DELTA
AS
SELECT
    a.account_id,
    a.workspace_id,
    a.run_by AS user_id,

    t.catalog_name,
    t.schema_name,
    t.table_name,

    t.privilege_type,
    t.permission_category,

    a.action_name,
    a.event_time,

    CURRENT_TIMESTAMP() AS load_timestamp

FROM data_governance.silver_access_management.audit_logs a
LEFT JOIN data_governance.silver_access_management.table_permissions t
    ON a.run_by = t.grantee